In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path
import time

dbalchemy = create_engine(f"postgresql+psycopg2://postgres@localhost:5432/dns_mac")

In [2]:
def get(day, hour0, hour1):
    query = f"""
    WITH 
    WINDOWS AS (
        SELECT id, dn_id, is_r, rcode, macsrc, macdst, FLOOR(SECONDS/3600) as hour FROM message3_it2016_0
        WHERE FLOOR(SECONDS/3600) BETWEEN {hour0} AND {hour1}
    ),
    TMP AS (
        SELECT
        {day} AS "day",
        hour AS "hour",
        dn.dac_family_rank1 AS DAC_FAMILY,
        
        COUNT(*) AS QR,
        COUNT(*) FILTER (WHERE IS_R IS FALSE) AS Q,
        COUNT(*) FILTER (WHERE RCODE = 0) AS OK,
        COUNT(*) FILTER (WHERE RCODE = 3) AS NX,
        COUNT(*) FILTER (WHERE NOT dn.regex_check) AS QR_NOTVALID,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check) AS Q_NOTVALID,
        COUNT(*) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check) AS NX_NOTVALID,
        COUNT(*) FILTER (WHERE RN=1) AS FA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1) AS FA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS FA_NX,
        COUNT(*) FILTER (WHERE RN_MAC=1) AS MACFA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1) AS MACFA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS MACFA_NX,


        COUNT(DISTINCT DN_ID) FILTER (WHERE P1) AS  P1_DN,
        COUNT(DISTINCT DN_ID) FILTER (WHERE NOT dn.regex_check AND P1) AS  P1_DN_NOTVALID,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=0 AND P1) AS  P1_DN_OK,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND P1) AS  P1_DN_NXD,
        COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND NOT dn.regex_check AND P1) AS  P1_DN_NXD_NOTVALID,
        COUNT(*) FILTER (WHERE P1) AS  P1_QR,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND P1) AS  P1_Q,
        COUNT(*) FILTER (WHERE RCODE = 0 AND P1) AS  P1_OK,
        COUNT(*) FILTER (WHERE RCODE = 3 AND P1) AS  P1_NX,
        COUNT(*) FILTER (WHERE NOT dn.regex_check AND P1) AS  P1_QR_NOTVALID,
        COUNT(*) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check AND P1) AS  P1_Q_NOTVALID,
        COUNT(*) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check AND P1) AS  P1_NX_NOTVALID,
        COUNT(*) FILTER (WHERE RN=1 AND P1) AS  P1_FA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1 AND P1) AS  P1_FA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1 AND P1) AS  P1_FA_NX,
        COUNT(*) FILTER (WHERE RN_MAC=1 AND P1) AS  P1_MACFA,
        COUNT(*) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1 AND P1) AS  P1_MACFA_OK,
        COUNT(*) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1 AND P1) AS  P1_MACFA_NX,

        SUM(LOGIt1) AS LLR1_QR,
        SUM(LOGIt1) FILTER (WHERE IS_R IS FALSE) AS LLR1_Q,
        SUM(LOGIt1) FILTER (WHERE RCODE = 0) AS LLR1_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE = 3) AS LLR1_NX,
        SUM(LOGIt1) FILTER (WHERE NOT dn.regex_check) AS LLR1_QR_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE IS_R IS FALSE AND NOT dn.regex_check) AS LLR1_Q_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE RCODE = 3 AND NOT dn.regex_check) AS LLR1_NX_NOTVALID,
        SUM(LOGIt1) FILTER (WHERE RN=1) AS LLR1_FA,
        SUM(LOGIt1) FILTER (WHERE RCODE=0 AND RN_QR_RCODE=1) AS LLR1_FA_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE=3 AND RN_QR_RCODE=1) AS LLR1_FA_NX,
        SUM(LOGIt1) FILTER (WHERE RN_MAC=1) AS LLR1_MACFA,
        SUM(LOGIt1) FILTER (WHERE RCODE=0 AND RN_MAC_QR_RCODE=1) AS LLR1_MACFA_OK,
        SUM(LOGIt1) FILTER (WHERE RCODE=3 AND RN_MAC_QR_RCODE=1) AS LLR1_MACFA_NX

        FROM WINDOWS M
        JOIN bigdn_m3_3 DN ON M.DN_ID=DN.ID
        JOIN M3_RN ON M3_RN.id=M.ID
        GROUP BY "day", hour, DAC_FAMILY
    )
    SELECT tmp.* FROM tmp
    """
    dfright = pd.read_sql(query, dbalchemy)
    # if Path('/tmp/dfright.csv').exists():
    #     dfright = pd.read_csv('/tmp/dfright.csv', index_col=0)
    # else:
    #     dfright = pd.read_sql(query, dbalchemy)
    #     dfright.to_csv('/tmp/dfright.csv')
    #     pass
    dflefts = []
    for idx, row in dfright.iterrows():
        dflefts.append(pd.read_sql(f"""select * from get_hourdn_statistics('{row['dac_family']}', {row['hour']}::int)""", dbalchemy))
    return pd.concat([pd.concat(dflefts).reset_index(drop=True), dfright.drop(columns='dac_family')], axis=1)

In [46]:

hstep = 2
for day in range(10):
    dfs = []
    for h in range(0, 24 - hstep + 1, hstep):
        s = time.time()
        h0, h1 = h, h + hstep - 1
        print(day, h0, h1, end=' ')
        fn = f'hourly/day{day}_{h0}_{h1}.csv'
        if Path(fn).exists():
            df = pd.read_csv(fn, index_col=0)
        else:
            df = get(day, h, h1) # BETWEEN is INCLUSIVE
        df.to_csv(fn)
        dfs.append(df)
        print(f'{time.time() - s}s')
        pass
    pd.concat(dfs, ignore_index=True).to_csv(f'daily/day{day}.csv')
    pass


0 0 1 0.004935026168823242s
0 2 3 0.0023398399353027344s
0 4 5 0.0018770694732666016s
0 6 7 0.0018661022186279297s
0 8 9 0.0020411014556884766s
0 10 11 0.0018160343170166016s
0 12 13 0.0018601417541503906s
0 14 15 0.0017290115356445312s
0 16 17 0.0015208721160888672s
0 18 19 0.0014400482177734375s
0 20 21 0.0013720989227294922s
0 22 23 0.0016980171203613281s
1 0 1 0.0016078948974609375s
1 2 3 0.0012972354888916016s
1 4 5 0.0012280941009521484s
1 6 7 0.0013298988342285156s
1 8 9 0.0011830329895019531s
1 10 11 0.0011670589447021484s
1 12 13 0.0011110305786132812s
1 14 15 0.0010612010955810547s
1 16 17 0.0010979175567626953s
1 18 19 0.0010650157928466797s
1 20 21 0.00096893310546875s
1 22 23 0.0009710788726806641s
2 0 1 0.0010020732879638672s
2 2 3 0.0010030269622802734s
2 4 5 0.0010061264038085938s
2 6 7 0.0008919239044189453s
2 8 9 0.0008909702301025391s
2 10 11 0.0009629726409912109s
2 12 13 0.0009288787841796875s
2 14 15 0.0008807182312011719s
2 16 17 0.0008728504180908203s
2 18 19 0.

In [48]:

col_id = [
    'day',
    'hour',
    'dac_family'
]
col_dn = [
    'dn_count',
    'dn_notvalid_count',
    'dn_ok_count',
    'dn_nxd_count',
    'dn_nxd_notvalid_count'
]
col_p1_dn = [ 'p1_' + c for c in col_dn ]

col_total = [
    'qr',
    'q',
    'ok',
    'nx',
    'qr_notvalid',
    'q_notvalid',
    'nx_notvalid',
    'fa',
    'fa_ok',
    'fa_nx',
    'macfa',
    'macfa_ok',
    'macfa_nx'
]

col_p1 = [ 'p1_'+c for c in col_total]
col_llr = [ 'llr1_'+c for c in col_total]

col_malicious = col_total + col_p1 + col_llr
col_features = ['hour'] + col_dn + col_total + col_p1 + col_llr
hstep = 2
for day in range(10):
    print(day)
    df = pd.read_csv(f'daily/without_p1_dn/day{day}.csv', index_col=0)
    df = df.drop(columns='hour.1')
    joins = []
    for idxrow, row in df.iterrows():
        print('\t', idxrow)
        joins.append(pd.read_sql(f"""select * from get_hourdn_statistics('{row['dac_family']}', {row['hour']}::int)""", dbalchemy))
        pass
    df_join = pd.concat(joins)
    dfjoined = df.merge(df_join, left_on=['hour', 'dac_family'], right_on=['hour', 'dac_family'], suffixes=('','_y')).drop(columns=[c + '_y' for c in col_dn])
    dfjoined.to_csv(f'daily/day{day}.csv')
    pass

0
	 0


NameError: name 'dbalchemy' is not defined

In [134]:
import itertools
import pandas as pd

col_id = ['dac_family'
    'day'
    'hour'
]
col_dn = [
    'dn_count',
    'dn_notvalid_count',
    'dn_ok_count',
    'dn_nxd_count',
    'dn_nxd_notvalid_count'
]
col_dn += [ 'p1_' + c for c in col_dn ]
col_total = [
    'qr',
    'q',
    'ok',
    'nx',
    'qr_notvalid',
    'q_notvalid',
    'nx_notvalid',
    'fa',
    'fa_ok',
    'fa_nx',
    'macfa',
    'macfa_ok',
    'macfa_nx'
]
col_p1 = [ 'p1_' + c for c in col_total]
col_llr = [ 'llr1_' + c for c in col_total]

col_numeric = col_dn + col_total + col_p1 + col_llr
col_malicious = col_total + col_p1 + col_llr

col_features = ['hour'] + col_dn + col_total + col_p1 + col_llr


df = pd.concat([ pd.read_csv(f'daily/day{d}.csv', index_col=0) for d in range(10) ])
df = df.drop(columns='p1_dn,p1_dn_notvalid,p1_dn_ok,p1_dn_nxd,p1_dn_nxd_notvalid'.split(','))
mws = [ item[0] for item in df[['dac_family']].drop_duplicates().to_numpy().tolist() ]
mws.remove('healthy')
df = df.set_index(['day','hour','dac_family'])

for c in df.columns:
    if c not in col_numeric:
        print(c)

singles = ([], [])
combined = ([], [])
labels = ([], [])
for d, h, mw in itertools.product(list(range(10)), list(range(24)), mws):
    healthyrow = df.loc[(d, h, 'healthy'), :].copy()
    lc = 1
    singles[0].append((d, h, mw))
    try:
        tmp = df.loc[(d, h, mw), :].copy()
        tmp[col_malicious] += (healthyrow[col_malicious])
        singles[1].append(pd.Series(df.loc[(d, h, mw), :].to_numpy(), index=df.columns))
        combined[1].append(pd.Series(tmp.copy().to_numpy(), index=df.columns))
    except KeyError as e:
        singles[1].append(pd.Series([ 0 for _ in col_numeric], index=df.columns))
        combined[1].append(pd.Series(healthyrow.to_numpy(), index=df.columns))
        values = healthyrow.copy()
        lc = 0
        pass
    labels[0].append((d, h, mw))
    labels[1].append(lc)
    labels[0].append((d, h, f'healthy+{mw}'))
    labels[1].append(lc)
    combined[0].append((d, h, f'healthy+{mw}'))
    pass

for d, h in itertools.product(list(range(10)), list(range(24))):
    singles[0].append((d,h,'healthy'))
    singles[1].append(df.loc[(d, h, 'healthy'), :].copy())
    pass

print(df.shape[0], len(combined), len(singles), df.shape[0] + len(combined) + len(singles), 2 * 240 * len(mws) + 240)

df = pd.DataFrame(combined[1] + singles[1], columns=df.columns, index=pd.MultiIndex.from_tuples(combined[0] + singles[0]))
df.insert(0, 'label', pd.Series(labels[1], index=labels[0]))
df['label'] = df['label'].fillna(0)
df.index.names = ['day', 'hour', 'dac_family']
df = df.sort_index(level=['day', 'hour', 'dac_family'])
df.to_csv('dataset.csv')



870 2 2 874 3120


In [ ]:

DATASET = pd.read_csv('dataset.csv', index_col=[0,1,2]).fillna(0)

DATASET.loc[pd.IndexSlice[(0,1), :, ('healthy', 'healthy+necurs', 'necurs')], :].sort_index(level=['day', 'hour', 'dac_family'])

label  dn_count  dn_notvalid_count  dn_ok_count  \
day hour dac_family                                                        
0   0    healthy           0.0   56058.0             2650.0      52642.0   
         healthy+necurs    1.0   60063.0             2650.0      54892.0   
         necurs            1.0   60063.0             2650.0      54892.0   
    1    healthy           0.0   47693.0             2112.0      45543.0   
         healthy+necurs    1.0   48522.0             2112.0      45992.0   
...                        ...       ...                ...          ...   
1   22   healthy+necurs    1.0   66526.0             3435.0      57760.0   
         necurs            1.0   66526.0             3435.0      57760.0   
    23   healthy           0.0   59299.0             3132.0      54038.0   
         healthy+necurs    1.0   63292.0             3132.0      56114.0   
         necurs            1.0   63292.0             3132.0      56114.0   

                         dn_nxd_count  dn_nxd_notvalid_count          qr  \
day hour dac_family                                                        
0   0    healthy               2829.0                 1296.0  16758780.0   
         healthy+necurs        4453.0                 1296.0  17446150.0   
         necurs                4453.0                 1296.0    687370.0   
    1    healthy               2003.0                 1076.0  13123300.0   
         healthy+necurs        2367.0                 1076.0  13213410.0   
...                               ...                    ...         ...   
1   22   healthy+necurs        6024.0                 1664.0  22869870.0   
         necurs                6024.0                 1664.0   1787880.0   
    23   healthy               3316.0                 1532.0  16562161.0   
         healthy+necurs        4963.0                 1532.0  17000795.0   
         necurs                4963.0                 1532.0    438634.0   

                                  q          ok         nx  ...  llr1_fa_ok  \
day hour dac_family                                         ...               
0   0    healthy          8032350.0  15999260.0   473260.0  ...         inf   
         healthy+necurs   8224340.0  16339270.0   785100.0  ...         inf   
         necurs            191990.0    340010.0   311840.0  ...         inf   
    1    healthy          6240910.0  12683780.0   317300.0  ...         0.0   
         healthy+necurs   6266540.0  12729190.0   359030.0  ...         0.0   
...                             ...         ...        ...  ...         ...   
1   22   healthy+necurs  10049550.0  20422720.0  1468660.0  ...         inf   
         necurs            421070.0    771720.0   820200.0  ...         inf   
    23   healthy          7741155.0  15547807.0   532841.0  ...         inf   
         healthy+necurs   7849427.0  15744061.0   730060.0  ...         inf   
         necurs            108272.0    196254.0   197219.0  ...         inf   

                         llr1_fa_nx  llr1_macfa  llr1_macfa_ok  llr1_macfa_nx  \
day hour dac_family                                                             
0   0    healthy                inf         inf            inf            inf   
         healthy+necurs         inf         inf            inf            inf   
         necurs                 inf         inf            inf            inf   
    1    healthy                inf         0.0            0.0            inf   
         healthy+necurs         inf         0.0            0.0            inf   
...                             ...         ...            ...            ...   
1   22   healthy+necurs         inf         inf            inf            inf   
         necurs                 inf         inf            inf            inf   
    23   healthy                inf         inf            inf            inf   
         healthy+necurs         inf         inf            inf            inf   
         necurs                 inf         inf

In [133]:

df.loc[pd.IndexSlice[:, :, ('healthy', 'healthy+conficker')], :]

# df.index
df.reset_index()['dac_family'].drop_duplicates().to_numpy().tolist()

['conficker',
 'healthy',
 'healthy+conficker',
 'healthy+modpack',
 'healthy+necurs',
 'healthy+pitou',
 'healthy+suppobox',
 'healthy+virut',
 'modpack',
 'necurs',
 'pitou',
 'suppobox',
 'virut']